# 04 — Running a CMIP7 diagnostic locally

The earlier notebooks read *pre-computed* results from the public API. Here we run a REF
diagnostic locally against a tiny **synthetic** CMIP7-style dataset, to see the evaluation
pipeline end to end. This example is for learning; it is not a real model evaluation.

**Prerequisites:** [01 — REF concepts](01-ref-concepts.ipynb).

**What you need:** the `climate-ref` packages. No data download is required.

## 1. Create the sample data

`create_cmip7_sample_data` writes small, synthetic `tas` and `areacella` NetCDF files
using the CMIP7 directory and filename layout. The files are generated locally so this
notebook can run in Binder without fetching a large model dataset.

In [1]:
from ref_tutorials import create_cmip7_sample_data

sample_data_dir = create_cmip7_sample_data()
sample_data_dir

PosixPath('data/cmip7-sample/MIP-DRS7/CMIP7/CMIP')

## 2. A custom diagnostic provider

Providers can be used directly, without the REF database or web infrastructure.

The tutorials define a minimal provider in `src/ref_tutorials/provider.py`.
Its diagnostic, `annual-mean-global-mean-tas`, computes the annual-mean, area-weighted
global-mean near-surface air temperature (`tas`) from CMIP7 data.

Open `src/ref_tutorials/provider.py` alongside this notebook. A diagnostic defines:

- `data_requirements` (what data it needs);
- `execute` (the calculation);
- `build_execution_result` (packaging the output).

In [2]:
from climate_ref.config import Config

from ref_tutorials.provider import provider

config = Config.default()
provider.configure(config)

diagnostic = provider.get("annual-mean-global-mean-tas")
diagnostic.slug

'annual-mean-global-mean-tas'

## 3. Build a data catalog from disk

A **data catalog** lists available datasets and their metadata. We build one from the
synthetic files using REF's CMIP7 dataset adapter. The generated data includes one
`tas` dataset and its matching `areacella` cell-area dataset.

In [3]:
from climate_ref.datasets import get_dataset_adapter

adapter = get_dataset_adapter("cmip7")
data_catalog = adapter.find_local_datasets(sample_data_dir)

data_catalog[["source_id", "variant_label", "variable_id", "experiment_id"]].drop_duplicates()

,source_id,variant_label,variable_id,experiment_id
0,Tutorial-Model,r1i1p1f1,areacella,historical
1,Tutorial-Model,r1i1p1f1,tas,historical


## 4. Solve for executions

Given the diagnostic's data requirements and the available datasets, the **solver**
works out which *executions* are possible. Each execution pairs the diagnostic with one
group of datasets.

In [4]:
from climate_ref.solver import solve_executions
from climate_ref_core.datasets import SourceDatasetType

executions = list(
    solve_executions(
        data_catalog={SourceDatasetType.CMIP7: data_catalog},
        diagnostic=diagnostic,
        provider=provider,
    )
)
print(f"{len(executions)} executions proposed")

1 executions proposed


## 5. Run one execution

An execution needs an `ExecutionDefinition` — it says which datasets to use and where to write output. 
We build one for the first execution and run the diagnostic directly.

In [5]:
from pathlib import Path

output_directory = Path("output") / "local-run"
definition = executions[0].build_execution_definition(output_directory)
definition.output_directory.mkdir(parents=True, exist_ok=True)

result = diagnostic.run(definition=definition)
assert result.successful, "diagnostic run failed"
print("Execution successful")

Execution successful


## 6. Inspect the output

The run wrote an output bundle into the output directory. Let's read it back.

In [6]:
import json

output_file = definition.to_output_path("output.json")
print(json.dumps(json.loads(output_file.read_text()), indent=2)[:2000])

{
  "index": "index.html",
  "provenance": {
    "environment": {},
    "modeldata": [],
    "obsdata": {},
    "log": "cmec_output.log"
  },
  "data": {},
  "plots": {
    "timeseries.png": {
      "filename": "timeseries.png",
      "long_name": "Annual-mean global-mean tas timeseries",
      "description": "Annual-mean global-mean tas timeseries",
      "dimensions": {
        "experiment_id": "historical",
        "source_id": "Tutorial-Model",
        "variant_label": "r1i1p1f1"
      }
    },
    "anomaly.png": {
      "filename": "anomaly.png",
      "long_name": "Annual-mean tas anomaly relative to the period mean",
      "description": "Annual-mean tas anomaly relative to the period mean",
      "dimensions": {
        "experiment_id": "historical",
        "source_id": "Tutorial-Model",
        "variant_label": "r1i1p1f1"
      }
    }
  },
  "html": {},
  "metrics": null,
  "diagnostics": {}
}


## Recap

- A **custom provider** runs a CMIP7 diagnostic locally, without the REF database.
- `find_local_datasets` builds a catalog from the generated CMIP7-style files.
- `solve_executions` pairs the diagnostic with matching datasets.
- `build_execution_definition` and `diagnostic.run` execute it.

Replace the synthetic files with real CMIP7 `tas` and `areacella` data to explore your
own model. See the [REF documentation](https://climate-ref.org) and
[API documentation](https://api.climate-ref.org/docs) for more detail.